# NumJa GPU POC - TornadoVM 5.2.0-jdk21 on NVIDIA T4

Runs `bench.tornadopoc.GemmBench`: CPU baseline (production HAL path, EJML) vs GPU (TornadoVM TaskGraph).
All shell work is `%%bash`; only the NumPy reference cell uses Python (numpy computes in-kernel).

**Runtime:** Menu > Runtime > Change runtime type > **T4 GPU** > Ubuntu 22.04. **Run all:** Ctrl+F9.
Step 1 is idempotent. Benchmark size N lives in `/tmp/bench-n.txt` (default 4096).


## Step 1 - Setup: JDK 21, Maven, TornadoVM SDK, CUDA 12.6, repo, build (one idempotent cell)

CUDA 12.6 workaround: Colab ships driver 580.x / CUDA 13, but TornadoVM 5.2.0-jdk21 binaries
target CUDA 12 (issue #710). Env (JAVA_HOME, PATH, LD_LIBRARY_PATH) is written to `/tmp/poc-env.sh`
and sourced by later cells (each `%%bash` cell is a fresh shell).

`ParallelRegressionTest` is skipped: a Phase-2 timing gate with known shared-CPU variance, unrelated to this POC.


In [1]:
%%bash
set -e
echo "=== GPU + Java ==="
nvidia-smi | head -8 || echo "WARNING: nvidia-smi failed - is this a GPU runtime?"
java -version 2>&1 | grep -v Cgroup

# --- JDK 21 (skip if present) ---
if ! java -version 2>&1 | grep -q '"21'; then
    apt-get update -qq && DEBIAN_FRONTEND=noninteractive apt-get install -y -qq openjdk-21-jdk
fi
export JAVA_HOME=$(dirname $(dirname $(readlink -f $(which java))))
echo "JAVA_HOME=$JAVA_HOME"

# --- Maven 3.9.15 (skip if present) ---
MVN_DIR=/opt/maven-3.9.15
if [ ! -d "$MVN_DIR" ]; then
    wget -q https://archive.apache.org/dist/maven/maven-3/3.9.15/binaries/apache-maven-3.9.15-bin.tar.gz -O /tmp/mvn.tgz
    tar -xzf /tmp/mvn.tgz -C /opt/
    mv /opt/apache-maven-3.9.15 "$MVN_DIR"
fi
export PATH="$MVN_DIR/bin:$PATH"
mvn --version 2>/dev/null | head -1

# --- TornadoVM 5.2.0-jdk21 SDK (skip if present) ---
SDK_DIR=/opt/tornadovm
if [ ! -d "$SDK_DIR" ]; then
    wget -q https://github.com/beehive-lab/TornadoVM/releases/download/v5.2.0-jdk21/tornadovm-5.2.0-jdk21-cuda-linux-amd64.tar.gz -O /tmp/tornado.tgz
    tar -xzf /tmp/tornado.tgz -C /opt/
    extracted=$(ls -d /opt/tornadovm* | head -1)
    [ "$extracted" = "$SDK_DIR" ] || mv "$extracted" "$SDK_DIR"
fi
echo "TornadoVM SDK: $SDK_DIR"

# --- CUDA 12.6 toolkit (TornadoVM targets CUDA 12; Colab driver is CUDA 13) ---
CUDA12_LIB=$(ls -d /usr/local/cuda-12.6/lib64 /usr/lib/cuda-12.6/lib64 2>/dev/null | head -1 || true)
[ -z "$CUDA12_LIB" ] && CUDA12_LIB=$(ls -d /usr/local/cuda-12*/lib64 /usr/lib/cuda-12*/lib64 2>/dev/null | head -1 || true)
echo "Chosen CUDA 12 lib: ${CUDA12_LIB:-none}"
if [ -z "$CUDA12_LIB" ]; then
    wget -q https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64/cuda-keyring_1.1-1_all.deb -O /tmp/cuda-keyring.deb
    dpkg -i /tmp/cuda-keyring.deb
    apt-get update -qq
    DEBIAN_FRONTEND=noninteractive apt-get install -y -qq --no-install-recommends cuda-toolkit-12-6
    CUDA12_LIB=$(ls -d /usr/local/cuda-12*/lib64 /usr/lib/cuda-12*/lib64 2>/dev/null | head -1 || true)
fi
if [ -n "$CUDA12_LIB" ]; then
    echo "CUDA 12 lib: $CUDA12_LIB"
    ls "$CUDA12_LIB"/libcudart* | head -2
else
    echo "WARNING: CUDA 12.6 toolkit not found; GPU run may fail (UnsatisfiedLinkError)"
fi

# --- persist env for later %%bash cells (fresh shells) ---
echo "export JAVA_HOME=$JAVA_HOME" > /tmp/poc-env.sh
echo "export PATH=$MVN_DIR/bin:\$PATH" >> /tmp/poc-env.sh
echo "export LD_LIBRARY_PATH=$CUDA12_LIB:\$LD_LIBRARY_PATH" >> /tmp/poc-env.sh

# --- clone repo + checkout phase-5 branch ---
REPO=/content/java_ml
BRANCH=gsd/phase-05-hardware-abstraction-layer-gpu-poc
if [ -d "$REPO/.git" ]; then
    git -C "$REPO" fetch origin "$BRANCH"
    git -C "$REPO" reset --hard "origin/$BRANCH"
else
    git clone --branch "$BRANCH" --single-branch https://github.com/minhhhduc/jml.git "$REPO"
fi
git -C "$REPO" log -1 --oneline

# --- install TornadoVM jars into ~/.m2 + parent POM stub ---
cd "$REPO"
SDK_JARS=$SDK_DIR/share/java/tornado
for art in tornado-api tornado-runtime tornado-drivers-cuda tornado-drivers-common; do
    mvn -q install:install-file -Dfile=$SDK_JARS/${art}-5.2.0-jdk21.jar -DgroupId=io.github.beehive-lab -DartifactId=$art -Dversion=5.2.0-jdk21 -Dpackaging=jar
    echo "OK $art"
done
PARENT_DIR=$HOME/.m2/repository/io/github/beehive-lab/tornado-drivers/5.2.0-jdk21
mkdir -p "$PARENT_DIR"
cat > "$PARENT_DIR/tornado-drivers-5.2.0-jdk21.pom" <<'POM'
<?xml version="1.0" encoding="UTF-8"?>
<project xmlns="http://maven.apache.org/POM/4.0.0">
    <modelVersion>4.0.0</modelVersion>
    <groupId>io.github.beehive-lab</groupId>
    <artifactId>tornado-drivers</artifactId>
    <version>5.2.0-jdk21</version>
    <packaging>pom</packaging>
</project>
POM
echo "parent POM written"

# --- build: numja install, tests (skip flaky timing gate), POC shaded jar ---
mvn -q -pl modules/numja -am install -DskipTests
mvn -pl modules/numja -am test -Dtest='!ParallelRegressionTest' 2>&1 | grep -E 'Tests run:|BUILD' | tail -4 || true
mvn -q -pl bench/tornado-poc -am package -DskipTests

echo "=== SETUP DONE ==="
ls -la bench/tornado-poc/target/tornado-poc-jar.jar
echo 4096 > /tmp/bench-n.txt


=== GPU + Java ===
Thu Sep 10 04:31:05 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
openjdk version "21.0.12" 2026-07-21
OpenJDK Runtime Environment (build 21.0.12+8-1-24.04-Ubuntu)
OpenJDK 64-Bit Server VM (build 21.0.12+8-1-24.04-Ubuntu, mixed mode, sharing)
JAVA_HOME=/usr/lib/jvm/java-21-openjdk-amd64
[0.000s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' s

Cloning into '/content/java_ml'...


## Step 2 - CPU baseline + GPU run on T4 + nvidia-smi telemetry (one bash cell)

CPU run: no `-Dtornado.device` -> production HAL path (EJML). GPU run: `-Dtornado.device=nvidia:0:0`.
`--enable-preview` on both: TornadoVM classes target the JDK 21 preview Vector API.
nvidia-smi samples util/mem every 1s during the GPU run; first sample (pre-run idle) is dropped.


In [2]:
%%bash
set -e
source /tmp/poc-env.sh
cd /content/java_ml
N=$(cat /tmp/bench-n.txt)
JAR=bench/tornado-poc/target/tornado-poc-jar.jar

echo "=== CPU baseline (N=$N) ==="
java --enable-preview -cp $JAR -Dbench.env=colab -Dbench.size=$N bench.tornadopoc.GemmBench 2>&1 | tee /tmp/poc-cpu.log

echo "=== GPU run (nvidia:0:0, N=$N) + nvidia-smi telemetry ==="
rm -f /tmp/nv-sample.log
nvidia-smi --query-gpu=utilization.gpu,memory.used --format=csv,noheader,nounits -l 1 -f /tmp/nv-sample.log &
NVPID=$!
sleep 0.5
java --enable-preview -cp $JAR -Dtornado.device=nvidia:0:0 -Dbench.env=colab -Dbench.size=$N bench.tornadopoc.GemmBench 2>&1 | tee /tmp/poc-gpu.log
kill $NVPID 2>/dev/null || true
sleep 0.5

# Telemetry: nvidia-smi CSV separator is ", ". Drop row 1 (pre-run idle); report peak/mean.
awk -F', ' 'NR>1 { n++; su+=$1; sm+=$2; if ($1>pu) pu=$1; if ($2>pm) pm=$2 } END { if (n>0) { printf "gpu_telemetry_samples=%d\n", n; printf "gpu_util_peak_pct=%.0f\n", pu; printf "gpu_util_mean_pct=%.1f\n", su/n; printf "gpu_mem_peak_mb=%.0f\n", pm; printf "gpu_mem_mean_mb=%.0f\n", sm/n } else print "gpu_telemetry_samples=0 (sampler failed)" }' /tmp/nv-sample.log || true


## Step 3 - NumPy magnitude reference (only Python cell: numpy computes in-kernel)

Java `util.Random` and NumPy PCG64 produce different data from the same seed values, so the NumPy
product is a **magnitude reference only**. The authoritative accuracy check is GemmBench's internal
full-matrix Frobenius metric (`cpu_vs_gpu_frob_rel_err`).


In [3]:
import numpy as np
N = int(open("/tmp/bench-n.txt").read().strip())
A_np = np.random.default_rng(0xC0FFEE).random((N, N))
B_np = np.random.default_rng(0xBADF00D).random((N, N))
C_np = A_np @ B_np
print(f"N={N}")
print(f"||C_np||_F = {np.linalg.norm(C_np, 'fro'):.6e}")
print(f"max |C_np| = {np.abs(C_np).max():.6e}")
print("Note: magnitude reference only (Java Random vs NumPy PCG64).")


N=4096
||C_np||_F = 4.194407e+06
max |C_np| = 1.094033e+03
Note: magnitude reference only (Java Random vs NumPy PCG64).


## Step 4 - Decision summary (bash)

Reads the key=value output from `/tmp/poc-cpu.log` and `/tmp/poc-gpu.log`. Capture this block for
`docs/05-GPU-POC-RESULTS.md`.


In [4]:
%%bash
echo "=== CPU run key=value ==="
grep -E '^(env|device|jdk|hardware|tornado.device|size|cpu_baseline_ms|result|verdict)=' /tmp/poc-cpu.log || echo '(no cpu log)'
echo "=== GPU run key=value ==="
grep -E '^(env|device|jdk|hardware|tornado.device|size|cpu_baseline_ms|gpu_ms|transfer_ms|speedup_ratio|transfer_pct|cpu_vs_gpu_[a-z_]+|result|verdict)=' /tmp/poc-gpu.log || echo '(no gpu log)'

verdict=$(grep -oP 'verdict=\K\w+' /tmp/poc-gpu.log || true)
frob=$(grep -oP 'cpu_vs_gpu_frob_rel_err=\K\S+' /tmp/poc-gpu.log || true)
speedup=$(grep -oP 'speedup_ratio=\K\S+' /tmp/poc-gpu.log || true)
echo
echo "Auto-verdict:          ${verdict:-?}"
echo "CPU vs GPU Frobenius:  ${frob:-?}"
echo "Speedup (CPU/GPU):     ${speedup:-?}"
case "$verdict" in
    GO)    echo "Recommendation: GO - pursue GPU backend milestone" ;;
    NO-GO) echo "Recommendation: NO-GO - defer GPU backend; investigate bottleneck" ;;
    *)     echo "INSUFFICIENT_DATA - check /tmp/poc-gpu.log and STDERR above" ;;
esac


=== CPU run key=value ===
cpu_baseline_ms=76787.0
result=GPU_ABSENT
verdict=NO-GO
env=colab
tornado.device=unset
jdk=21.0.12
hardware=Linux+amd64+2cores
size=4096
device=cpu-thread
=== GPU run key=value ===
cpu_baseline_ms=77365.0
device=GPU_ABSENT
gpu_ms=NA
transfer_ms=NA
speedup_ratio=NA
transfer_pct=NA
result=DEVICE_MISMATCH
verdict=NO-GO
env=colab
tornado.device=nvidia:0:0
jdk=21.0.12
hardware=Linux+amd64+2cores
size=4096

Auto-verdict:          NO
CPU vs GPU Frobenius:  ?
Speedup (CPU/GPU):     NA
INSUFFICIENT_DATA - check /tmp/poc-gpu.log and STDERR above


## Decision rubric

**Numerical (must hold):** `cpu_vs_gpu_frob_rel_err <= 1e-9` (else `result=NUMERIC_MISMATCH` - kernel bug).
Expected ~1e-12 for double-precision GEMM at N=4096 (O(N*eps)).

**Performance (go/no-go):** `speedup_ratio >= 2.0` AND `transfer_pct < 50.0` -> **GO**; otherwise **NO-GO**.

**Failure labels:** `GPU_ABSENT` (no -Dtornado.device), `DEVICE_MISMATCH` (HW-03 guard),
`GPU_INIT_FAILED` (native/driver error), `NUMERIC_MISMATCH` (kernel wrong output).
